# Deception Detection — Full Colab Pipeline

**Paper**: Audio-Visual Deception Detection (ICCV 2023) — PECL  
**Dataset**: DOLOS ("Would I Lie to You?")  

**Setup**: Runtime → Change runtime type → **T4 GPU**  

### What this notebook does:
1. Clones repo + installs dependencies
2. Mounts Google Drive (where your data lives)
3. Extracts face frames from raw videos using MTCNN (GPU-accelerated)
4. Trains Audio-Visual Fusion model with crossmodal attention
5. Saves models + logs back to Google Drive

---
## Step 1: Clone Repository & Install Dependencies

In [ ]:
# Clone the repo
!git clone https://github.com/PreciousTechnologies/Deception-Detection-using-Face-and-audio.git
%cd Deception-Detection-using-Face-and-audio

# Install all dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q opencv-python-headless scikit-learn pandas matplotlib seaborn tqdm pillow librosa facenet-pytorch

print("Done — all packages installed.")

## Step 2: Verify GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("NO GPU! Go to Runtime → Change runtime type → T4 GPU")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

---
## Step 3: Mount Google Drive

You should have already uploaded `DOLOS_upload.zip` to your Google Drive root.
This zip contains: `raw_videos/` (1427 .mp4) + `audio_files/` (1427 .wav)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify the zip exists on Drive
import os
zip_path = '/content/drive/MyDrive/DOLOS_upload.zip'
if os.path.exists(zip_path):
    size_gb = os.path.getsize(zip_path) / 1e9
    print(f"Found: {zip_path} ({size_gb:.2f} GB)")
else:
    print("\n*** FILE NOT FOUND ***")
    print("Please upload DOLOS_upload.zip to your Google Drive root folder.")
    print("Expected location: Google Drive → My Drive → DOLOS_upload.zip")
    print("\nTo create this zip locally, run in your project folder:")
    print('  powershell -File prepare_colab_upload.ps1')

## Step 4: Unzip Data from Google Drive

In [ ]:
import os, subprocess

# Create data directory structure
os.makedirs('data/audio_files', exist_ok=True)
os.makedirs('data/face_frames', exist_ok=True)
os.makedirs('data/protocols', exist_ok=True)

# Copy training protocols from repo
!cp DOLOSDATA/DOLOS/Training_Protocols/*.csv data/protocols/

# Unzip audio and raw videos from Drive
zip_path = '/content/drive/MyDrive/DOLOS_upload.zip'
if os.path.exists(zip_path):
    print("Unzipping... this may take a few minutes.")
    !unzip -qo "{zip_path}" -d data/
    print("Unzip complete.")
else:
    raise FileNotFoundError(f"{zip_path} not found. Upload it to Google Drive first.")

# Verify contents
audio_count = len([f for f in os.listdir('data/audio_files') if f.endswith('.wav')]) if os.path.exists('data/audio_files') else 0
video_dir = 'data/raw_videos'
if not os.path.exists(video_dir):
    # Try alternate paths the zip might use
    for alt in ['data/DOLOSDATA/raw_videos', 'data/raw_videos']:
        if os.path.exists(alt):
            video_dir = alt
            break

video_count = len([f for f in os.listdir(video_dir) if f.endswith('.mp4')]) if os.path.exists(video_dir) else 0
print(f"\nAudio .wav files: {audio_count}")
print(f"Raw .mp4 videos: {video_count}")
print(f"Video dir path:  {video_dir}")

---
## Step 5: Extract Face Frames (GPU-accelerated MTCNN)

This processes all 1,427 videos on the T4 GPU. ~10-15 minutes.

Each video produces a folder of face-cropped JPG frames.

In [ ]:
import cv2
import numpy as np
from PIL import Image
from facenet_pytorch import MTCNN
from tqdm.notebook import tqdm

# Detect video directory
video_dir = None
for candidate in ['data/raw_videos', 'data/DOLOSDATA/raw_videos']:
    if os.path.exists(candidate) and len([f for f in os.listdir(candidate) if f.endswith('.mp4')]) > 0:
        video_dir = candidate
        break

if video_dir is None:
    raise FileNotFoundError("No raw_videos directory found!")

face_out_dir = 'data/face_frames'
os.makedirs(face_out_dir, exist_ok=True)

# Initialize MTCNN on GPU
device = torch.device('cuda:0')
mtcnn = MTCNN(keep_all=True, device=device, min_face_size=40)

videos = sorted([f for f in os.listdir(video_dir) if f.endswith('.mp4')])
print(f"Processing {len(videos)} videos on {device}...")

skipped = 0
extracted = 0
failed = 0

for v in tqdm(videos, desc="Extracting faces"):
    clip_name = v.replace('.mp4', '')
    frame_dir = os.path.join(face_out_dir, clip_name)

    # Skip if already extracted
    if os.path.exists(frame_dir) and len(os.listdir(frame_dir)) >= 10:
        skipped += 1
        continue

    os.makedirs(frame_dir, exist_ok=True)

    try:
        cap = cv2.VideoCapture(os.path.join(video_dir, v))
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        # Sample ~4 frames per second
        sample_interval = max(1, int(fps // 4))
        frame_idx = 0
        saved_count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            if frame_idx % sample_interval == 0:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                img = Image.fromarray(frame_rgb)

                # Detect and crop face
                face_tensor = mtcnn(img)
                if face_tensor is not None:
                    # Resize to 160x160 and save
                    face_pil = torch.nn.functional.interpolate(
                        face_tensor.unsqueeze(0), size=(160, 160), mode='bilinear'
                    ).squeeze(0)
                    face_pil = (face_pil * 255).byte().cpu()
                    face_pil = Image.fromarray(face_pil.permute(1, 2, 0).numpy())
                    face_pil.save(os.path.join(frame_dir, f'{saved_count:04d}.jpg'))
                    saved_count += 1

            frame_idx += 1
        cap.release()

        if saved_count > 0:
            extracted += 1
        else:
            failed += 1

    except Exception as e:
        failed += 1
        if failed <= 3:
            print(f"  Failed: {v} — {e}")

print(f"\nDone!")
print(f"  Extracted: {extracted}")
print(f"  Skipped (already done): {skipped}")
print(f"  Failed: {failed}")
print(f"  Total face folders: {len(os.listdir(face_out_dir))}")

---
## Step 6: Verify Dataset is Ready

In [ ]:
import pandas as pd

# Check protocols
train_df = pd.read_csv('data/protocols/train_fold1.csv')
test_df = pd.read_csv('data/protocols/test_fold1.csv')
print(f"Train samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Columns: {list(train_df.columns)}")
print(f"\nLabel distribution:")
print(train_df.iloc[:, 1].value_counts())

# Count audio and face frames
audio_files = [f for f in os.listdir('data/audio_files') if f.endswith('.wav')]
face_dirs = [d for d in os.listdir('data/face_frames') if os.path.isdir(os.path.join('data/face_frames', d))]

print(f"\nAudio files: {len(audio_files)}")
print(f"Face frame folders: {len(face_dirs)}")

# Check overlap with protocol
train_names = set(train_df.iloc[:, 0].values)
test_names = set(test_df.iloc[:, 0].values)
all_names = train_names | test_names

missing_audio = all_names - set([f.replace('.wav', '') for f in audio_files])
missing_faces = all_names - set(face_dirs)

print(f"\nMissing audio for protocol: {len(missing_audio)}")
print(f"Missing face frames for protocol: {len(missing_faces)}")
if missing_faces:
    print(f"  First 5 missing: {list(missing_faces)[:5]}")

---
## Step 7: Train AV Fusion Model (Single Fold)

PECL architecture: Wav2Vec2 + ViT-B16 + Crossmodal Attention + Efficient Conv Adapters

In [ ]:
import sys, os
sys.path.insert(0, 'scripts')

from train_test import train_test
from types import SimpleNamespace
import torch

config = SimpleNamespace(
    device='cuda:0',
    device_ID='cuda:0',
    lr=1e-3,
    batch_size=16,
    num_epochs=30,
    seed=2222,
    num_encoders=4,
    adapter=True,
    adapter_type='efficient_conv',
    data_root='data/',
    audio_path='data/audio_files/',
    visual_path='data/face_frames/',
    log='logs',
    protocols=[['train_fold1.csv', 'test_fold1.csv']],
    model_name='DOLOS_',
    model_to_train='fusion',
    fusion_type='cross2',
    multi=False,
)

torch.manual_seed(config.seed)
config.device = torch.device(config.device_ID)

os.makedirs(config.log, exist_ok=True)
log_name = os.path.join(config.log, f'colab_fusion_seed{config.seed}.txt')
with open(log_name, 'w') as f:
    f.write(f"Optimizer: Adam, LR: {config.lr}\n")
    f.write(f"Batch Size: {config.batch_size}\n")
    f.write(f"Epochs: {config.num_epochs}\n")
    f.write(f"Encoders: {config.num_encoders}\n")
    f.write(f"Adapter: {config.adapter}, Type: {config.adapter_type}\n")
    f.write(f"Fusion: {config.fusion_type}\n")
    f.write("-" * 40 + "\n")

print("Starting fusion training...")
print(f"  LR={config.lr}, Batch={config.batch_size}, Epochs={config.num_epochs}")
print(f"  Fusion={config.fusion_type}, Adapter={config.adapter_type}")

train_test(log_name, config)

---
## Step 8: 3-Fold Cross-Validation

In [ ]:
import sys, os
sys.path.insert(0, 'scripts')

from train_test import train_test
from types import SimpleNamespace
import torch

for fold in range(1, 4):
    print(f"\n{'='*60}")
    print(f"  FOLD {fold}/3")
    print(f"{'='*60}")

    config = SimpleNamespace(
        device='cuda:0',
        device_ID='cuda:0',
        lr=1e-3,
        batch_size=16,
        num_epochs=30,
        seed=2222 + fold,
        num_encoders=4,
        adapter=True,
        adapter_type='efficient_conv',
        data_root='data/',
        audio_path='data/audio_files/',
        visual_path='data/face_frames/',
        log='logs',
        protocols=[[f'train_fold{fold}.csv', f'test_fold{fold}.csv']],
        model_name='DOLOS_',
        model_to_train='fusion',
        fusion_type='cross2',
        multi=False,
    )

    torch.manual_seed(config.seed)
    config.device = torch.device(config.device_ID)

    os.makedirs(config.log, exist_ok=True)
    log_name = os.path.join(config.log, f'colab_fold{fold}.txt')
    with open(log_name, 'w') as f:
        f.write(f"Fold {fold}\nLR: {config.lr}\nBatch: {config.batch_size}\nEpochs: {config.num_epochs}\n")

    train_test(log_name, config)

print("\n" + "="*60)
print("ALL FOLDS COMPLETE")
print("="*60)

for fold in range(1, 4):
    log_file = os.path.join('logs', f'colab_fold{fold}.txt')
    if os.path.exists(log_file):
        print(f"\n--- Fold {fold} ---")
        with open(log_file) as f:
            print(f.read())

---
## Step 9: Visualize Training Curves

In [ ]:
import matplotlib.pyplot as plt
import re

def parse_log(path):
    tr_acc, te_acc, tr_auc, te_auc = [], [], [], []
    with open(path) as f:
        for line in f:
            m = re.search(
                r'train_acc ([\d.]+).*train_auc:([\d.]+).*test_acc ([\d.]+).*test_auc:([\d.]+)', line)
            if m:
                tr_acc.append(float(m.group(1)))
                te_acc.append(float(m.group(3)))
                tr_auc.append(float(m.group(2)))
                te_auc.append(float(m.group(4)))
    return tr_acc, te_acc, tr_auc, te_auc

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for fold in range(1, 4):
    lf = os.path.join('logs', f'colab_fold{fold}.txt')
    if os.path.exists(lf):
        ta, tea, trau, tear = parse_log(lf)
        if ta:
            axes[0].plot(tea, label=f'Fold {fold}', linewidth=2)
            axes[1].plot(tear, label=f'Fold {fold}', linewidth=2)

axes[0].set_title('Test Accuracy per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Test AUC per Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 10: Save Everything to Google Drive

In [ ]:
import shutil

drive_out = '/content/drive/MyDrive/DeceptionDetection_Results'
os.makedirs(drive_out, exist_ok=True)

# Copy logs
if os.path.exists('logs'):
    shutil.copytree('logs', os.path.join(drive_out, 'logs'), dirs_exist_ok=True)

# Copy plots
for f in ['training_curves.png']:
    if os.path.exists(f):
        shutil.copy(f, drive_out)

# Copy model weights
if os.path.exists('logs'):
    for f in os.listdir('logs'):
        if f.endswith('.pth'):
            shutil.copy(os.path.join('logs', f), drive_out)

print(f"Saved to: {drive_out}")
!ls -lh "{drive_out}"